In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
from openai import OpenAI
import os

In [2]:
# Load the CSV into a DataFrame
acct_names_df = pd.read_csv('Account Names3.csv')

In [3]:
# Step 1: Load the CSV file into a DataFrame
# Replace 'your_file.csv' with the actual path to your CSV file


# Step 2: Define a function to perform fuzzy matching
# The function returns a match only if the score is above a threshold, otherwise it returns an empty string
def fuzzy_match(row, col1, col2, threshold=80):
    if pd.isna(row[col2]):  # If the SalesNav Name is missing, return empty values
        return "", 0
    
    match, score, _ = process.extractOne(row[col1], acct_names_df[col2], scorer=fuzz.ratio)
    
    if score >= threshold:  # Only consider it a match if the score exceeds the threshold
        return match, score
    else:
        return "", 0

# Step 3: Apply the fuzzy matching function
# This will add two new columns: 'Best Match' (the best fuzzy match from Account Name) and 'Match Score' (the similarity score)
acct_names_df[['Deterministic Match', 'Match Score']] = acct_names_df.apply(fuzzy_match, axis=1, col1='SalesNav Name', col2='Account Name', result_type='expand')

# Step 4: Display the resulting DataFrame
print(acct_names_df)

# Optionally, save the result to a new CSV file
acct_names_df.to_csv('fuzzy_matched_results.csv', index=False)

           SalesNav Name                    Account Name Deterministic Match  \
0        & Other Stories                      Grado Labs   And Other Stories   
1     Ahold Delhaize USA                       Zojirushi  Ahold Delhaize USA   
2              ALVIC USA             Zippo Manufacturing            ALDI USA   
3              Arc'teryx                             Zep           Arc'teryx   
4               Armarkat                            Zagg            Armarkat   
...                  ...                             ...                 ...   
2838                   -          Great American Cookies                       
2839                   -                Koss Corporation                       
2840                   -                         Seagate                       
2841                   -                    ButterflyMX®                       
2842                   -  Recaro Child Safety (Consumer)                       

      Match Score  
0       87.500000  

In [6]:
OpenAI.api_key = os.environ["OPENAI_API_KEY"]

In [9]:
# Function to match company names from a DataFrame with two columns
def match_company_names(df, col1, col3):
    # Extract the relevant columns
    names_col1 = df[col1].tolist()
    names_col3 = df[col3].tolist()

    # Construct a prompt with all the names in column 1 and column 3
    prompt = f"Match the following company names from List1 with their best possible matches from List2:\n\n"
    for i, name1 in enumerate(names_col1):
        prompt += f"{i+1}. {name1}\n"
    prompt += "\nList2:\n"
    for j, name3 in enumerate(names_col3):
        prompt += f"{j+1}. {name3}\n"
    
    prompt += "\nFor each entry in List1, provide the best match from List2 by returning the corresponding entry from List2. If no suitable match is found, return 'No match'."

    client = OpenAI()
    # Send the request to ChatGPT
    response = client.completions.create(
        model="gpt-3.5-turbo-instruct",
        prompt=prompt,
        max_tokens=500
    )
    
    # Parse the response (assuming it returns something like "1: 3, 2: 1, 3: No match")
    match_results = response.choices[0].text.strip().split("\n")

    # Initialize a list to store the best match for each entry in column 1
    best_matches = []

    # Process the output from ChatGPT and append the matches to the DataFrame
    for result in match_results:
        if "No match" in result:
            best_matches.append("-")
        else:
            # Extract the index from the response and convert it to the corresponding company in column 3
            idx_col3 = int(result.split(":")[1].strip()) - 1
            best_matches.append(names_col3[idx_col3])

    # Add the match results as a new column to the DataFrame
    df['Best Match'] = best_matches
    
    return df

In [10]:
# Call the matching function
acct_names_df['LLM Match'] = match_company_names(acct_names_df, 'SalesNav Name', 'Account Name')['LLM Match']

# Print the matched companies
acct_names_df_v2.to_csv('fuzzy_matched_results_v2.csv', index=False)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

SyntaxError: invalid syntax (2463506640.py, line 1)